# Volume Dataset Unified Benchmark (Greedy + Full + Stochastic)

This notebook runs all solvers with one unified interface:

`VolumeDataset -> VolumeSolver.solve_checked() -> AssignmentSolution -> VolumeDataset.evaluate()`


In [ ]:
from pathlib import Path
import sys

REPO_ROOT_BOOT = Path('/Users/igoreshka/Desktop/Optimization-of-flows').resolve()
SRC_PATH = REPO_ROOT_BOOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
print('Using SRC_PATH:', SRC_PATH)


In [ ]:
from __future__ import annotations

import gzip
import json
import shutil
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from flowopt.volume_core import (
    GreedyBatchConfig,
    GreedyBatchVolumeSolver,
    VolumeDataset,
    VolumeGapVRPLikeSolver,
    VolumeMilpLikeSolver,
    VolumeGeneticLikeSolver,
    VolumeGapVRPStochasticSolver,
    VolumeMilpStochasticSolver,
    VolumeGeneticStochasticSolver,
    assert_dataset_input,
    assert_solution_output,
    save_solution_artifacts,
)


In [ ]:
REPO_ROOT = Path('/Users/igoreshka/Desktop/Optimization-of-flows').resolve()
DATA_DIR = REPO_ROOT / 'demo/data/object_volume_feasible_fullfleet/full_all_tasks_all_agents_stage4_volume_only'

CANDIDATE_DATASETS = [
    DATA_DIR / 'dataset_real_spb_clean_full_all_tasks_all_agents_stage4_volume_only_greedy_full_subset_with_distances_v2_100.json',
    DATA_DIR / 'dataset_real_spb_clean_full_all_tasks_all_agents_stage4_volume_only_greedy_full_subset_with_distances.json',
    DATA_DIR / 'dataset_real_spb_clean_full_all_tasks_all_agents_stage4_volume_only_feasible_subset_with_distances.json',
    DATA_DIR / 'dataset_real_spb_clean_full_all_tasks_all_agents_stage4_volume_only.json',
]

DATASET_PATH = next((p for p in CANDIDATE_DATASETS if p.exists()), CANDIDATE_DATASETS[-1])
DATASET_GZ_PATH = (DATA_DIR / 'dataset_real_spb_clean_full_all_tasks_all_agents_stage4_volume_only.json.gz')

OUT_ROOT = REPO_ROOT / 'demo/local/volume_unified_7algo'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUT_ROOT / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Profiles:
# - quality: closer to feasible/full coverage
# - fast: speed-biased stochastic behavior with possible coverage drop
PROFILE = 'quality'  # 'quality' | 'fast'

if PROFILE == 'quality':
    MAX_RUNTIME_SEC = 120.0
    TOP_K_AGENTS = 30
    TOP_K_DESTINATIONS = 4
    MAX_TASKS_IN_TRIP = 500
else:
    MAX_RUNTIME_SEC = 70.0
    TOP_K_AGENTS = 30
    TOP_K_DESTINATIONS = 4
    MAX_TASKS_IN_TRIP = 500

LOG_EVERY_SEC = 20.0
TRIP_LOG_EVERY = 400
SAVE_DETAILED_ARTIFACTS = False

print('PROFILE:', PROFILE)
print('REPO_ROOT:', REPO_ROOT)
print('DATASET_PATH:', DATASET_PATH)
print('RUN_DIR:', RUN_DIR)


In [ ]:
if not DATASET_PATH.exists():
    if not DATASET_GZ_PATH.exists():
        raise FileNotFoundError(f'Neither dataset json nor json.gz exists: {DATASET_PATH}')
    print('Unpacking base dataset from .json.gz ...')
    with gzip.open(DATASET_GZ_PATH, 'rb') as src, DATASET_PATH.open('wb') as dst:
        shutil.copyfileobj(src, dst)
    print('Unpacked:', DATASET_PATH)

assert DATASET_PATH.exists(), 'Dataset json must exist after unpacking step.'


In [ ]:
base_cfg = GreedyBatchConfig(
    max_runtime_sec=MAX_RUNTIME_SEC,
    top_k_agents=TOP_K_AGENTS,
    top_k_destinations=TOP_K_DESTINATIONS,
    max_tasks_in_trip=MAX_TASKS_IN_TRIP,
    log_every_sec=LOG_EVERY_SEC,
    trip_log_every=TRIP_LOG_EVERY,
    verbose=True,
)

# Keep deterministic solvers in full-feasible mode by default.
# Stochastic solvers are included for speed/coverage trade-off.
SOLVERS = [
    (
        'greedy_full',
        lambda: GreedyBatchVolumeSolver(
            GreedyBatchConfig(
                max_runtime_sec=MAX_RUNTIME_SEC,
                top_k_agents=TOP_K_AGENTS,
                top_k_destinations=TOP_K_DESTINATIONS,
                max_tasks_in_trip=MAX_TASKS_IN_TRIP,
                log_every_sec=LOG_EVERY_SEC,
                trip_log_every=TRIP_LOG_EVERY,
                verbose=True,
                score_mode='vol_per_km',
                stochastic_mode=False,
                deterministic_fill=True,
                fill_time_budget_sec=60.0 if PROFILE == 'quality' else 20.0,
            )
        ),
    ),
    ('gap_vrp_like', lambda: VolumeGapVRPLikeSolver(base_cfg)),
    ('milp_like', lambda: VolumeMilpLikeSolver(base_cfg)),
    ('genetic_like', lambda: VolumeGeneticLikeSolver(base_cfg)),
    ('gap_vrp_stoch', lambda: VolumeGapVRPStochasticSolver(base_cfg)),
    ('milp_stoch', lambda: VolumeMilpStochasticSolver(base_cfg)),
    ('genetic_stoch', lambda: VolumeGeneticStochasticSolver(base_cfg)),
]

print('Solvers:', [x[0] for x in SOLVERS])


In [ ]:
rows = []
artifacts = {}

for name, solver_factory in SOLVERS:
    print('\n' + '='*20 + f' {name.upper()} START ' + '='*20)
    dataset = VolumeDataset.from_json(DATASET_PATH)
    assert_dataset_input(dataset)

    solver = solver_factory()

    t0 = time.perf_counter()
    if hasattr(solver, 'solve_checked'):
        solution = solver.solve_checked(dataset)
    else:
        solution = solver.solve(dataset)
        solution = assert_solution_output(solution, dataset_path=str(dataset.dataset_path))
    wall = time.perf_counter() - t0

    evaluation = dataset.evaluate(solution)

    out_dir = RUN_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    if SAVE_DETAILED_ARTIFACTS:
        artifacts[name] = save_solution_artifacts(
            dataset=dataset,
            solution=solution,
            evaluation=evaluation,
            out_dir=out_dir,
        )
    else:
        sol_path = out_dir / 'solution.json'
        eval_path = out_dir / 'evaluation.json'
        sol_path.write_text(json.dumps({
            'algorithm': solution.algorithm,
            'dataset_path': solution.dataset_path,
            'runtime_sec': solution.runtime_sec,
            'unassigned_task_ids': list(solution.unassigned_task_ids),
            'trips_count': len(solution.trips),
        }, ensure_ascii=False, indent=2), encoding='utf-8')
        eval_path.write_text(json.dumps(evaluation.as_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
        artifacts[name] = {'solution_json': str(sol_path), 'evaluation_json': str(eval_path)}

    row = evaluation.as_dict()
    row['solver_name'] = name
    row['wall_runtime_sec'] = round(float(wall), 3)
    row['quality_feasible_flag'] = 1.0 if bool(row.get('feasible')) else 0.0
    rows.append(row)

    print(
        f"{name}: assigned={row['assigned_tasks']}/{row['total_tasks']} "
        f"coverage={row['task_coverage_pct']}% feasible={row['feasible']} "
        f"solver_runtime={row['runtime_sec']}s wall={row['wall_runtime_sec']}s"
    )

summary = pd.DataFrame(rows)
summary


In [ ]:
# Unified quality metric for ranking
# Higher is better.

s = summary.copy()

cov = s['task_coverage_pct'].astype(float) / 100.0
feas = s['feasible'].astype(bool).astype(float)
wall = s['wall_runtime_sec'].astype(float)

# Min-max normalize for cost-like metrics (lower is better)
def inv_minmax(x: pd.Series) -> pd.Series:
    x = x.astype(float)
    lo, hi = float(x.min()), float(x.max())
    if hi - lo < 1e-12:
        return pd.Series(np.ones(len(x)), index=x.index)
    return 1.0 - (x - lo) / (hi - lo)

runtime_score = inv_minmax(wall)
km_score = inv_minmax(s['total_km'].fillna(s['total_km'].max()))
tr_score = inv_minmax(s['transport_work_volume_m3_km'].fillna(s['transport_work_volume_m3_km'].max()))

s['quality_score'] = (
    0.55 * cov
    + 0.20 * feas
    + 0.15 * runtime_score
    + 0.05 * km_score
    + 0.05 * tr_score
)

sort_cols = ['quality_score', 'task_coverage_pct', 'feasible', 'wall_runtime_sec']
s = s.sort_values(sort_cols, ascending=[False, False, False, True]).reset_index(drop=True)

summary_path_csv = RUN_DIR / 'summary_all_algorithms.csv'
summary_path_json = RUN_DIR / 'summary_all_algorithms.json'
s.to_csv(summary_path_csv, index=False)
summary_path_json.write_text(s.to_json(orient='records', force_ascii=False, indent=2), encoding='utf-8')

print('Saved:', summary_path_csv)
print('Saved:', summary_path_json)

view_cols = [
    'solver_name', 'algorithm', 'feasible', 'all_checks_ok',
    'assigned_tasks', 'unassigned_tasks', 'task_coverage_pct',
    'active_agents', 'trips_count', 'transport_work_volume_m3_km',
    'total_km', 'total_hours', 'runtime_sec', 'wall_runtime_sec', 'quality_score'
]
s[view_cols]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

plot_df = s.copy()
labels = plot_df['solver_name'].tolist()

axes[0].bar(labels, plot_df['task_coverage_pct'])
axes[0].set_title('Coverage %')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(labels, plot_df['wall_runtime_sec'])
axes[1].set_title('Wall Runtime (sec)')
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(labels, plot_df['quality_score'])
axes[2].set_title('Unified Quality Score')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
print('Run dir:', RUN_DIR)
print('Artifacts index sample:')
for k, v in artifacts.items():
    print(k, '->', v.get('solution_json', 'n/a'))
